# darts_forecast — diagnostics: the frozen config on TEST, the stress tests, the one-shot HOLDOUT, the verdict

The other three notebooks *choose* things: the inputs (`darts_forecast_data`), the forecaster
(`darts_forecast_model`) and the trading rule (`darts_forecast`), all of it on **TUNE**. This notebook chooses
nothing. It takes ONE frozen configuration and scores it on data that played no part in choosing it:

* **TEST** (2026-02-01 → 2026-04-30) is scored here: the forecast quality, the backtest against buy-and-hold,
  the stress tests and the coin-flip null. (`darts_forecast.ipynb` can replay TEST too, behind `READ_TEST = True`;
  neither look may change a knob);
* **HOLDOUT** (2026-05-01 → the last stored bar) stays unread unless you set `READ_HOLDOUT = True` — once, after the
  TEST verdict is in and nothing about the candidate will change any more.

**This run RE-READS TEST; it is not a fresh one-shot.** TEST was first scored on 2026-10-02 under the older forecast
core. An audit then fixed that core's bugs: the grid now follows the clock and masks half-days, each option source
has its own "has data" flag, a target never spans two sessions, the trade path no longer swallows signals, and the
forecast cache is keyed by the panel's full content. `config.yaml` changed with the fixes (`gbm_iter` 200 → 100,
`risk_per_trade` 0.005 → 0.001). The numbers below are TEST read a second time, after those fixes. So TEST has been
seen: a knob changed because of what it shows would make it one more tuning window. Only the HOLDOUT is still unread.

"Frozen" means the **knobs**: model, grid, inputs, retraining cadence, trading rule. It does not mean the model's
weights. The walk-forward simply continues through TEST: at every retrain point (before every session by default)
the model is refit on the previous `train_days` sessions, as it would be live, so the training window gradually
fills with earlier TEST sessions. That is allowed — a model only ever trains on bars before the ones it forecasts.
What is not allowed is changing a knob after looking at TEST.

| § | what | reads |
|---|---|---|
| 0 | the knobs: which candidate, the holdout switch, what to run | — |
| 1 | the split, and the frozen configuration | — |
| 2 | load through the end of TEST, continue the walk-forward, check that TEST changes nothing in TUNE | TUNE + TEST |
| 3 | **forecast quality on TEST**, next to TUNE: IC, hit rate, calibration, by month, name and hour | TEST |
| 4 | **the backtest on TEST** against equal-weight buy-and-hold: equity, metrics, per month, the trades | TEST |
| 5 | **stress**: pessimistic fills, 2× costs, no costs, the health gate off / on, long only, short only | TEST |
| 6 | **the coin-flip null** on TEST: the whole pipeline on a market without direction, judged by `rs.null_z` | TEST (surrogate) |
| 7 | the **HOLDOUT**, read once — off by default | HOLDOUT |
| 8 | **checks, the verdict** (four TEST gates) and one `SCORECARD` line | — |

**How to read the numbers.** The forecast is judged by its **rank IC**: the rank correlation between the forecast and
the realized return over the next `horizon` grid bars (0 = no information), with a *t* clustered by day because the
five names share each day's market. The book is judged like every strategy in this repo, against equal-weight
buy-and-hold of the same names (`strategies/baseline`): **alpha** (`alpha_cagr` = CAGR − beta × buy-and-hold CAGR) is
the study's one objective — the return left after the market exposure the book actually ran. **Excess**
(`excess_cagr`, the return over owning the five names) is shown too, but a book that is flat every night and
long/short during the day cannot earn the basket's return by construction, so it is not gated.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --all-extras  (Darts is the optional "darts" extra)
import dataclasses
import functools
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import text

from trading_simple import backtest as bt, optimize as opt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds what the darts_forecast notebooks share (tables, backtests, metrics), charts.py
# the figure helpers; signals() never imports them.
_spec = importlib.util.spec_from_file_location(
    "darts_research", strategies.STRATEGIES_DIR / "darts_forecast" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
_spec = importlib.util.spec_from_file_location(
    "darts_charts", strategies.STRATEGIES_DIR / "darts_forecast" / "charts.py"
)
charts = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(charts)
strat, SM = rs.STRAT, rs.SM  # the loaded strategy and strategy.py's own functions
baseline = strategies.load("baseline")  # equal-weight buy-and-hold: the repo's hurdle
TUNE, TEST, HOLDOUT = rs.WINDOWS["tune"], rs.WINDOWS["test"], rs.WINDOWS["holdout"]
INIT_CASH = settings.backtest.init_cash
COLOR = {"tune": plots._MUTED, "test": plots.SERIES[0], "holdout": plots.SERIES[4]}

# --- 0 · the knobs --------------------------------------------------------------------------------
CANDIDATE = "config"  # "config" = config.yaml as shipped; "pick" or a trial number = a row of the
#     candidates file darts_forecast.ipynb saves (studies/<STUDY>.candidates.yaml)
STUDY = None  # None = config.yaml's optuna.study; darts_forecast.ipynb appends its FORECAST
#     overrides to the study's name, so name that study here to diagnose one of its rows
OVERRIDES = {}  # another FORECAST config to diagnose, e.g. {"model": "tide", "bar_minutes": 15}.
#     Choose it on TUNE first (model notebook): a knob changed after reading TEST makes TEST one
#     more tuning window. A deep model's walk-forward through TEST takes hours without
#     finetune_epochs > 0
READ_HOLDOUT = False  # True ONLY when the TEST verdict is in and nothing will change; read it once
RUN_NULL = True  # §6: one full walk-forward per seed on a coin-flip market (cached on disk)
NULL_SEEDS = (0, 1)
RUN_TRUNCATION = True  # §2: the TUNE-only walk-forward (free when another notebook cached it)
COST_MULT = 2.0  # §5: every cost (commission, its minimum, spread, slippage) times this
MIN_IC_T = 2.0  # §8 gate: TEST rank IC > 0 with a day-clustered t >= this
MIN_NULL_Z = 2.0  # §8 gate: the TEST IC above the coin-flip runs by z >= this (rs.null_z)

## 1 · The split, and the frozen configuration (Table 1a, Table 1b)

The split lives in one place, `research.WINDOWS` (**Table 1a**). **TUNE** is everything a knob may be chosen on.
Inside TUNE the walk-forward is already out of sample: at each retrain point the model is **fit** on the previous
`train_days` sessions (the TRAIN slice; TiDE / TFT also hold back the last `val_days` of them as a VALIDATION slice
for early stopping) and then **forecasts** the bars up to the next retrain point. But every knob was chosen while
looking at TUNE, so a good TUNE number can be the product of that choice. **TEST** answers it: the same pipeline and
the same knobs, on months nobody chose anything on (read once before, under the old core: see the top).

**Table 1b** is the one configuration this notebook judges: its *forecast* knobs drive the walk-forward, its
*trading-rule* knobs turn the forecasts into trades. `CANDIDATE = "config"` is `config.yaml` as shipped. `"pick"` (the
study's own choice) or a trial number takes a row of the candidates file `darts_forecast.ipynb` saves after its
study, `studies/<STUDY>.candidates.yaml`: the forecast config the study ran on, plus that row's searched trading-rule
knobs. That study searches only the trading rule, on TUNE, for one objective (alpha).

In [ ]:
display(rs.windows_table())

STUDY_NAME = STUDY or strat.optuna["study"]
CAND_PATH = opt.study_path(strat, STUDY_NAME).with_name(f"{STUDY_NAME}.candidates.yaml")
PRM, SOURCE = rs.candidate(CANDIDATE, CAND_PATH)
PRM = {**PRM, **OVERRIDES}
assert strat.valid(PRM), "the candidate is not a valid parameter set (strategy.valid)"
CHANGED = {k: v for k, v in PRM.items() if strat.params.get(k) != v}
print(f"candidate: {SOURCE}" + (f" + OVERRIDES {OVERRIDES}" if OVERRIDES else ""))
print(f"differs from config.yaml in: {CHANGED or 'nothing'}")
frozen = pd.DataFrame(
    {
        "part": ["forecast" if k in SM["MODEL_KEYS"] else "trading rule" for k in PRM],
        "value": [str(v) for v in PRM.values()],
        "config.yaml": [str(strat.params.get(k)) for k in PRM],
    },
    index=pd.Index(list(PRM), name="knob"),
)
display(frozen.sort_values("part", kind="stable"))

## 2 · Load through the end of TEST and continue the walk-forward (Table 2a–2c, Fig 2)

The panel is loaded from the first stored session through the last day of TEST — `rs.load(settings, engine,
end=WINDOWS["test"][1])`, whose `end` is the last session, inclusive, so not one HOLDOUT bar is loaded — with the
option inputs attached, and the walk-forward runs over all of it. **Every TEST forecast comes from a model fit only
on sessions before it**, and every rolling input (the volatility scale, the daily inputs, the health gate) enters
TEST *warm*, with the history it would have had live.

`rs.prime_cache` reads that walk-forward back from `cache/` when it holds one for exactly these forecast knobs, this
panel's full content (every bar and option input) and this `strategy.py`. Otherwise it runs it — one fit per
session, **Table 2a**: `fits` × `mean_s`; ~10 minutes for `gbm`, hours for TiDE / TFT refit from scratch — and saves
it. A backfill that changed a stored bar makes a different key, so the forecast simply re-runs on the new bars.
`prime_cache` also hands the forecast to `signals()`, so every backtest below reuses it.

**Fig 2** is the split as a timeline: the warm-up sessions that only build inputs, then a few sample fits, each with
its TRAIN window (plus VALIDATION for TiDE / TFT with `val_days` > 0) and the sessions it forecasts (the diamond:
with the daily default a fit forecasts one session, a sliver at this scale). **Table 2b** is the option inputs'
coverage, TUNE against TEST, per source: each input, and the source's `has_*` flag (any of its inputs present). A
missing input enters the model as 0 with its source's flag off, so a source that thins out in TEST changes what the
model sees.

**Table 2c — the truncation check.** The TUNE part of this run must be *identical* — not close — to the walk-forward
on the panel that ends with TUNE, the one the model and strategy notebooks score (`rs.load` with its default `end`,
read back from `cache/`). The check first confirms that the two loads hold the same TUNE bars and option inputs,
then compares the inputs, the fits (retrain points, TRAIN windows, sample counts) and every TUNE forecast. Whatever
reads bars after the cut changes once TEST is appended: an input or a volatility scale normalized over the whole
sample, a fit that trains on later sessions. A look-ahead of a bar or two inside TUNE does not reach across the cut
and cannot show here; the strategy's tests pin that one (a forecast never reads an input after its decision bar).

In [ ]:
panel = rs.load(settings, engine, end=TEST[1])  # the first stored session -> TEST's last day
SESS = [str(d) for d in pd.Series(panel.close.index.tz_convert(panel.tz).date).unique()]
assert SESS[-1] <= TEST[1], "the panel reaches into the HOLDOUT"
RT_BP = rs.round_trip_bp(settings, panel)  # the engine's round trip at the $2,000 order size
with engine.connect() as conn:  # metadata only: the date of the last stored bar, no prices
    LAST_BAR = conn.execute(
        text("SELECT max(ts) FROM prices_ohlcv WHERE symbol = ANY(:s) AND bar = '1m'"),
        {"s": panel.symbols},
    ).scalar()
print(
    f"{len(panel.symbols)} names ({', '.join(panel.symbols)}) · "
    f"{len(panel.close):,} one-minute bars · {len(SESS)} sessions {SESS[0]} → {SESS[-1]} · "
    f"last stored bar {LAST_BAR:%Y-%m-%d}"
)
TEST_SESS = [s for s in SESS if s >= TEST[0]]
print(
    f"TEST sessions loaded: {len(TEST_SESS)}, {TEST_SESS[0]} → {TEST_SESS[-1]} "
    f"(TEST ends {TEST[1]})"
)
print(f"round trip at the ${strat.order_notional:,.0f} order size: {RT_BP:.1f} bp")

# the walk-forward through TEST: read back from cache/ (the key holds every forecast knob, this
# panel's content and strategy.py's version), or run now and saved; handed to signals() too
fc = rs.prime_cache(panel, PRM)
FC_OK = bool(
    fc["panel_id"] == SM["fingerprint"](panel)
    and fc.get("params", rs.model_params(PRM)) == rs.model_params(PRM)
)
FC_DETAIL = (
    f"panel_id {fc['panel_id']}: "
    + (
        "made from the loaded bars and option inputs with the candidate's forecast knobs"
        if FC_OK
        else "NOT the loaded panel's forecast, or other forecast knobs"
    )
    + ("; read back from cache/" if fc["model"] is None else "; computed in this run and saved")
)
print(f"forecast: {FC_DETAIL}")
LOG = fc["log"].assign(window=lambda d: np.where(d["session"] >= TEST[0], "TEST", "TUNE"))
SCORE_FROM = LOG["session"].iloc[0]
table2a = LOG.groupby("window", sort=False).agg(
    fits=("fit", "size"),
    first_retrain=("session", "min"),
    last_retrain=("session", "max"),
    mean_s=("seconds", "mean"),
    total_s=("seconds", "sum"),
)
display(table2a.round(1))

VAL = PRM["val_days"] if PRM["model"] in rs.SM["DEEP"] else 0
first = LOG.iloc[0]
test_log = LOG[LOG["window"] == "TEST"]
only_test = test_log[test_log["train_from"] >= TEST[0]]
print(
    f"warm-up (inputs only): {SESS[0]} → {rs.sess(first['train_from'], -1, SESS)}; "
    f"first fit: {rs.fit_windows(first, VAL, SESS)}; first forecast {SCORE_FROM}"
)
print(
    f"VALIDATION: the last {VAL} of the {PRM['train_days']} sessions of every window "
    "(early stopping only, never fitted)"
    if VAL
    else f"VALIDATION: none - {PRM['model']} (val_days {PRM['val_days']}) fits every window "
    "whole; val_days applies to tide / tft"
)
if len(test_log):
    f0 = test_log.iloc[0]
    msg = f"first TEST fit: {rs.fit_windows(f0, VAL, SESS)}, forecasts {f0['session']}"
    if len(only_test):
        msg += f" · TRAIN holds TEST sessions only from {only_test['session'].iloc[0]} on"
    print(msg)

# Fig 2 · the timeline
fig = go.Figure()
FC_NAME = {
    "tune": "TUNE: forecasts (out of sample, but seen)",
    "test": "TEST: forecasts scored here",
}
warm_end = rs.sess(first["train_from"], -1, SESS)
charts.span(fig, "TUNE", SESS[0], warm_end, plots._MUTED, "warm-up: inputs only", opacity=0.35)
charts.span(fig, "TUNE", first["train_from"], first["train_last"], plots.SERIES[1], "TRAIN")
if VAL > 0:
    charts.span(
        fig,
        "TUNE",
        rs.sess(first["train_last"], 1, SESS),
        rs.sess(SCORE_FROM, -1, SESS),
        plots.SERIES[2],
        "VALIDATION",
    )
charts.span(fig, "TUNE", SCORE_FROM, TUNE[1], COLOR["tune"], FC_NAME["tune"])
charts.span(fig, "TEST", TEST[0], SESS[-1], COLOR["test"], FC_NAME["test"])
charts.span(
    fig,
    "HOLDOUT",
    HOLDOUT[0],
    f"{LAST_BAR:%Y-%m-%d}",
    COLOR["holdout"],
    "HOLDOUT: read once" if READ_HOLDOUT else "HOLDOUT: not read",
    pattern=None if READ_HOLDOUT else "/",
    opacity=1.0 if READ_HOLDOUT else 0.45,
)
n_tune = int((LOG["window"] == "TUNE").sum())
sample = {0, n_tune // 2, max(n_tune - 1, 0), min(n_tune, len(LOG) - 1), len(LOG) - 1}
if len(only_test):
    sample.add(int(only_test.index[0]))
for i in sorted(sample):
    r = LOG.iloc[i]
    row = f"fit {int(r['fit']) + 1} · {r['session']}"
    charts.span(fig, row, r["train_from"], r["train_last"], plots.SERIES[1], "TRAIN")
    if VAL > 0:
        charts.span(
            fig,
            row,
            rs.sess(r["train_last"], 1, SESS),
            rs.sess(r["session"], -1, SESS),
            plots.SERIES[2],
            "VALIDATION",
        )
    nxt = LOG["session"].iloc[i + 1] if i + 1 < len(LOG) else None
    end = rs.sess(nxt, -1, SESS) if nxt and nxt > r["session"] else r["session"]
    w = "test" if r["session"] >= TEST[0] else "tune"
    charts.span(fig, row, r["session"], end, COLOR[w], FC_NAME[w])
    day0, day1 = pd.Timestamp(r["session"]), pd.Timestamp(end) + pd.Timedelta(days=1)
    fig.add_trace(  # a one-session forecast span is a sliver at this scale: mark it
        go.Scatter(
            x=[day0 + (day1 - day0) / 2],
            y=[row],
            mode="markers",
            marker={
                "symbol": "diamond",
                "size": 11,
                "color": COLOR[w],
                "line": {"color": plots._INK, "width": 1},
            },
            legendgroup=FC_NAME[w],
            showlegend=False,
            hovertemplate=f"fit {int(r['fit']) + 1} forecasts {r['session']} → {end}"
            "<extra></extra>",
        )
    )
fig.update_layout(barmode="overlay")
plots._base(
    fig, title="Fig 2 · the split: what each window and each sample fit is used for", height=460
)
fig.update_xaxes(type="date")
fig.update_yaxes(autorange="reversed")
charts.below(fig, -0.12)
fig.show()

# Table 2b · the option inputs' coverage, TUNE (scored part) vs TEST
table2b = pd.DataFrame(
    {
        "TUNE (scored)": rs.coverage(panel.features, PRM["options"], SCORE_FROM, TUNE[1]),
        "TEST": rs.coverage(panel.features, PRM["options"], *TEST),
    }
)
if len(table2b):
    table2b.index.names = ["source", "input"]
    display(table2b.round(2))
else:
    print(f"options = {PRM['options']!r}: the candidate reads no option inputs")

In [ ]:
# Table 2c · the truncation check: does appending TEST change anything in TUNE?
TRUNC_OK, TRUNC_DETAIL = None, "not run (RUN_TRUNCATION = False)"
if RUN_TRUNCATION:
    tune_panel = rs.load(settings, engine)  # through TUNE's last day: the other notebooks' panel
    n1 = len(tune_panel.close)
    same_bars = panel.close.index[n1] >= pd.Timestamp(TEST[0], tz="UTC") and all(
        getattr(tune_panel, k).equals(getattr(panel, k).iloc[:n1])
        for k in ("open_", "high", "low", "close", "volume")
    )
    same_options = set(tune_panel.features) == set(panel.features) and all(
        f.equals(panel.features[k].reindex(f.index)) for k, f in tune_panel.features.items()
    )
    fc_tune = rs.cached_forecast(tune_panel, PRM)  # the model / strategy notebooks' TUNE run
    a_in, b_in = fc["inputs"], fc_tune["inputs"]
    n0, H = len(b_in["g"]["code"]), a_in["horizon"]
    same_inputs = (
        a_in["past_names"] == b_in["past_names"]
        and all(
            np.array_equal(a_in["past"][s][:n0], b_in["past"][s], equal_nan=True)
            for s in panel.symbols
        )
        and all(np.array_equal(a_in[k][:n0], b_in[k], equal_nan=True) for k in ("target", "scale"))
        and np.array_equal(a_in["valid"][:n0], b_in["valid"])
    )
    cols = ["fit", "retrain_bucket", "session", "train_from", "train_last", "samples"]
    tune_fits = fc["log"].loc[fc["log"]["session"] <= TUNE[1], cols].reset_index(drop=True)
    same_fits = tune_fits.equals(fc_tune["log"][cols].reset_index(drop=True))
    a, b = fc["edge"][:n0], fc_tune["edge"]
    both = np.isfinite(a) & np.isfinite(b)
    one_only = int((np.isfinite(a) ^ np.isfinite(b))[: n0 - H].sum())  # the last H have no outcome
    n_diff = int((a[both] != b[both]).sum())
    max_diff = float(np.abs(a[both] - b[both]).max()) if both.any() else np.nan
    TRUNC_OK = bool(
        all([same_bars, same_options, same_inputs, same_fits, both.any()])
        and one_only == n_diff == 0
    )
    TRUNC_DETAIL = (
        f"{int(both.sum()):,} TUNE forecasts from {len(tune_fits)} fits compared: {n_diff} differ "
        f"(max |difference| {max_diff:.1e}), {one_only} in one run only; same bars and option "
        f"inputs {same_bars and same_options}, same inputs {same_inputs}, same fits {same_fits}"
    )
    display(
        pd.Series(
            {
                "same TUNE bars in both loads": same_bars,
                "same option inputs in both loads": same_options,
                "inputs identical (past, target, valid, scale)": same_inputs,
                "fits identical (retrain points, TRAIN windows, samples)": same_fits,
                "forecasts compared": int(both.sum()),
                "forecasts that differ": n_diff,
                "max |forecast difference|": max_diff,
                "forecasts in only one run": one_only,
                "identical": TRUNC_OK,
            },
            name="TEST appended vs TUNE only",
        ).to_frame()
    )
    if not TRUNC_OK:
        print(
            "the two loads hold different TUNE data (a backfill between them?): no verdict on "
            "look-ahead - re-run"
            if not (same_bars and same_options)
            else "NOT identical: an input or a fit reads bars after the cut"
        )
print(TRUNC_DETAIL)

## 3 · Forecast quality on TEST, next to TUNE (Table 3a–3c, Fig 3a–3c)

Every scored forecast is a (decision bar, name) pair whose outcome — the return over the next `horizon` grid bars —
completes inside the same session. **Table 3a** puts TEST next to TUNE:

| number | meaning | no information |
|---|---|---|
| `ic`, `t` | rank correlation of forecast and outcome; t clustered by day | 0 |
| `hit` | how often the forecast has the outcome's sign | 0.5 |
| `auc` | P(an up move is ranked above a down move) | 0.5 |
| `mse_skill` | 1 − MSE / MSE of always forecasting 0 (z units) | ≤ 0 |
| `sign_bp`, `sign_t` | mean of sign(forecast) × outcome, bp per grid bar, before costs | 0 |
| `net_bp` | `sign_bp` minus one 14 bp round trip at the real $2,000 order size | −14 |

`net_bp` trades every bar; the trading rule trades only the confident ones, so §4 is the real test of tradability.

**Fig 3a** calibration: the forecasts cut into deciles, the mean forecast (dotted) and the mean realized return (solid)
of each. An informative forecast rises from left to right, in both windows. **Fig 3b** the IC per month with ±2
standard errors (TEST in blue). **Fig 3c** the IC over a rolling 10 sessions, TEST shaded. **Tables 3b / 3c** break
TEST down by name and by hour of the day.

In [ ]:
t_tune, t_test = rs.table(fc, "tune"), rs.table(fc, "test")
S_TUNE, S_TEST = rs.summary(t_tune, cost_bp=RT_BP), rs.summary(t_test, cost_bp=RT_BP)
table3a = pd.DataFrame({"TUNE (scored)": S_TUNE, "TEST": S_TEST}).drop(["n", "days"])
display(table3a.astype(float).round(4))
print(
    f"scored forecasts: TUNE {S_TUNE['n']:,} on {S_TUNE['days']} sessions, "
    f"TEST {S_TEST['n']:,} on {S_TEST['days']} sessions; net_bp charges a {RT_BP:.1f} bp round trip"
)

fig = go.Figure()
for w, t in (("TUNE", t_tune), ("TEST", t_test)):
    c = rs.calibration(t)
    colr = COLOR[w.lower()]
    fig.add_trace(
        go.Scatter(
            x=c.index + 1,
            y=c["real_bp"],
            mode="lines+markers",
            name=f"{w}: realized",
            line={"color": colr, "width": 2},
        )
    )
    fig.add_trace(
        go.Scatter(
            x=c.index + 1,
            y=c["edge_bp"],
            mode="lines",
            name=f"{w}: forecast",
            line={"color": colr, "width": 1, "dash": "dot"},
        )
    )
fig.add_hline(y=0, line={"color": plots._INK, "dash": "dot", "width": 1})
plots._base(
    fig,
    title="Fig 3a · calibration: mean forecast and mean outcome per forecast decile",
    height=420,
)
fig.update_xaxes(title_text="forecast decile (1 = most bearish, 10 = most bullish)", dtick=1)
fig.update_yaxes(title_text="bp over the horizon")
charts.below(fig, -0.2)
fig.show()

MONTHS = rs.by(pd.concat([t_tune, t_test]), "month", cost_bp=RT_BP).astype(float)
MONTHS["window"] = np.where(MONTHS.index >= TEST[0][:7], "TEST", "TUNE")
se = (MONTHS["ic"] / MONTHS["t"]).abs()
fig = go.Figure()
for w in ("TUNE", "TEST"):
    m = MONTHS[MONTHS["window"] == w]
    fig.add_trace(
        go.Bar(
            x=m.index,
            y=m["ic"],
            name=w,
            marker_color=COLOR[w.lower()],
            error_y={"type": "data", "array": 2 * se[m.index], "color": plots._INK, "thickness": 1},
        )
    )
fig.add_hline(y=0, line={"color": plots._INK, "width": 1})
plots._base(
    fig, title="Fig 3b · rank IC per month, ±2 standard errors (clustered by day)", height=380
)
charts.below(fig, -0.15)
fig.show()

ric = rs.rolling_ic(pd.concat([t_tune, t_test]), days=10)
ric.index = pd.to_datetime(ric.index)
fig = go.Figure(
    plots._scatter(
        ric, name="rank IC, rolling 10 sessions", line={"color": plots.SERIES[0], "width": 1.5}
    )
)
fig.add_hline(y=0, line={"color": plots._INK, "dash": "dot", "width": 1})
fig.add_vrect(
    x0=TEST[0],
    x1=SESS[-1],
    fillcolor=COLOR["test"],
    opacity=0.08,
    line_width=0,
    layer="below",
    annotation_text="TEST",
    annotation_position="top left",
)
plots._base(fig, title="Fig 3c · the forecast's rank IC over time (TEST shaded)", height=360)
fig.update_xaxes(type="date")
charts.below(fig, -0.15)
fig.show()

KEEP = ["ic", "t", "hit", "auc", "sign_bp", "sign_t", "n"]
print("Table 3b · TEST by name")
display(rs.by(t_test, "name", cost_bp=RT_BP)[KEEP].astype(float).round(3))
print("Table 3c · TEST by hour of the decision (New York)")
display(rs.by(t_test, "hour", cost_bp=RT_BP)[KEEP].astype(float).round(3))

## 4 · The backtest on TEST, against buy-and-hold (Table 4a–4d, Fig 4a, Fig 4b)

The candidate's trading rule on the TEST forecasts, through the repo's one engine: `bt.evaluate(...,
score_from=WINDOWS["test"][0])`. `signals()` runs on the whole panel — so the forecasts, the volatility scale and the
health gate enter TEST warm — and the book is traded, and measured, from the first TEST bar on. How the rule trades
(`strategy.trade_frames`):

* a decision at a grid bar's close fills at the next one-minute bar's open and asks to hold its side `hold_bars` grid
  bars (0 = `horizon`). The book follows that **intended position path** per name: a new same-side decision extends
  the hold, an opposite one closes it and the new side opens one minute later, and the book is flat at the
  session's last real bar (`eod_exit`);
* the stop sits at `stop_vol` typical moves over the hold;
* each entry deploys `min(1 / max_positions, risk_per_trade / stop)` of the free cash of ONE shared $10k wallet. At
  `risk_per_trade` 0.001 the risk rule binds whenever the stop is wider than 0.5%: the trade then gets less than a
  slot, sized to lose 0.1% of the free cash at its stop (§4 prints how often that happens);
* costs are the engine's at the real $2,000 order size: 14 bp a round trip.

The hurdle is `strategies/baseline`, equal-weight buy-and-hold of the same five names over the same bars and costs.
A book that is flat overnight and long/short during the day has a beta near zero; it is not asked to earn the
basket's return, it is asked for **alpha** — something the basket does not earn.

**Table 4a** the book and the baseline. **Fig 4a** their cumulative return over TEST, on one scale; **Fig 4b** the
book alone, with its realized P&L against the fees it paid. **Table 4b** per month: the book, the baseline, the
trades and the forecast's IC. **Table 4c** the trades per side: how many, how often they win, the mean net return per
trade, how long they are held, the median order size, and how they end. The ending is read off the book's own signal
path: the hold ran out (time exit), an opposite decision closed it (reversal), the session's last bar, or the stop —
an exit the path did not plan, or a planned exit that did not fill at the open (when the stop is touched inside the
bar of a planned exit, the engine fills the stop, the worse price). **Table 4d** the P&L per name: an edge that lives
in one name is an anecdote.

In [ ]:
ST_BOOK = bt.evaluate(strat, panel, PRM, settings=settings, score_from=TEST[0])
ST_BASE = bt.evaluate(baseline, panel, {}, settings=settings, score_from=TEST[0])
pf_book = bt.run_strategy(strat, panel, PRM, settings=settings, score_from=TEST[0])
pf_base = bt.run_strategy(baseline, panel, {}, settings=settings, score_from=TEST[0])
COLS = [
    "cagr",
    "alpha_cagr",
    "beta",
    "excess_cagr",
    "info_ratio",
    "tracking_error",
    "volatility",
    "sharpe",
    "max_drawdown",
    "win_rate",
    "n_trades",
    "trades_per_year",
    "years",
]
table4a = pd.DataFrame({"darts_forecast": ST_BOOK, "buy & hold (baseline)": ST_BASE}).loc[COLS]
display(table4a.astype(float).round(4))

fig = go.Figure()
for pf, name, line in (
    (pf_base, "buy & hold (equal weight)", {"color": plots._MUTED, "dash": "dash", "width": 1.5}),
    (pf_book, "darts_forecast", {"color": plots.SERIES[0], "width": 1.5}),
):
    curve = bt.equity_curve(pf, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0
    fig.add_trace(plots._scatter(curve, name=name, line=line))
plots._base(fig, title=f"Fig 4a · cumulative return over TEST ({TEST[0]} → {SESS[-1]})", height=440)
fig.update_xaxes(type="date")
fig.update_yaxes(tickformat="+.0%")
charts.below(fig, -0.12)
fig.show()
fig = plots.plot_equity(
    bt.equity_curve(pf_book, init_cash=INIT_CASH),
    title="Fig 4b · the book alone: value, return, realized P&L against the fees paid",
)
charts.below(fig, -0.06)
fig.show()

TRADES = bt.trade_records(pf_book)
TRADES["month"] = TRADES["entry_ts"].dt.tz_convert(panel.tz).dt.strftime("%Y-%m")
by_month = TRADES.groupby("month")
table4b = pd.DataFrame(
    {
        "book": rs.monthly(pf_book, INIT_CASH, panel.tz),
        "buy & hold": rs.monthly(pf_base, INIT_CASH, panel.tz),
        "trades": by_month.size().astype(float),
        "win rate": by_month["pnl"].apply(lambda s: (s > 0).mean()),
        "P&L $": by_month["pnl"].sum(),
        "forecast IC": MONTHS["ic"],
        "IC t": MONTHS["t"],
    }
).loc[lambda d: d.index >= TEST[0][:7]]
display(table4b.fillna({"trades": 0, "P&L $": 0}).round(4))

# how each trade ended, read off the book's own signal path (the forecasts are cached)
SIG = strat.signals(panel, **PRM)
T0 = pd.Timestamp(TEST[0], tz="UTC")  # where bt.evaluate starts scoring
planned = {"Long": rs.marks(SIG["exits"]), "Short": rs.marks(SIG["short_exits"])}
opposite = {"Long": rs.marks(SIG["short_entries"]), "Short": rs.marks(SIG["entries"])}
following = dict(zip(panel.close.index[:-1], panel.close.index[1:], strict=True))
last_real = set(panel.close.index[fc["inputs"]["g"]["last_1m"]])
SLIP = bt.strategy_costs(strat, panel, settings)[1]  # an order at the open fills at open x (1±SLIP)
HOLD_MIN = (PRM["hold_bars"] or PRM["horizon"]) * PRM["bar_minutes"]
TRADES["exit"] = [
    rs.how_closed(r, panel.open_, SLIP, planned, opposite, following, last_real)
    for _, r in TRADES.iterrows()
]
TRADES = TRADES.assign(
    minutes=(TRADES["exit_ts"] - TRADES["entry_ts"]).dt.total_seconds() / 60,
    bp=1e4 * TRADES["return"],
    order=TRADES["size"].abs() * TRADES["entry_price"],
)
table4c = TRADES.groupby("direction").agg(
    trades=("pnl", "size"),
    win_rate=("pnl", lambda s: (s > 0).mean()),
    mean_bp=("bp", "mean"),
    median_minutes=("minutes", "median"),
    pnl=("pnl", "sum"),
    median_order=("order", "median"),
)
table4c = table4c.join(pd.crosstab(TRADES["direction"], TRADES["exit"], normalize="index"))
display(table4c.round(3))
N_TEST = len(TEST_SESS)
clear = (t_test["edge_bp"].abs() > PRM["edge_bp"]).mean()
print(
    f"{len(TRADES):,} trades on {TRADES['entry_ts'].dt.tz_convert(panel.tz).dt.date.nunique()} of "
    f"{N_TEST} TEST sessions; mean net return per trade {TRADES['bp'].mean():+.1f} bp (the costs, "
    f"a {RT_BP:.1f} bp round trip, are already netted in); planned hold {HOLD_MIN} min, "
    "longer when a same-side decision extends it"
)
print(
    f"what limits the trades: {clear:.1%} of the scored TEST forecasts clear edge_bp "
    f"({PRM['edge_bp']:g} bp), before the time-of-day filters and the health gate"
)
entry = (SIG["entries"] | SIG["short_entries"]).to_numpy() & (panel.close.index >= T0)[:, None]
frac, slot = SIG["cash_frac"].to_numpy()[entry], 1.0 / PRM["max_positions"]
if entry.any():
    print(
        f"size: min(slot {slot:.0%}, risk_per_trade {PRM['risk_per_trade']:g} / stop) of the free "
        f"cash; median {np.median(frac):.1%}, below the slot (the risk rule binds) on "
        f"{(frac < slot - 1e-12).mean():.0%} of the {int(entry.sum()):,} TEST entries"
    )
print("Table 4d · P&L ($) and trades per name and side")
display(
    TRADES.pivot_table(
        index="symbol", columns="direction", values="pnl", aggfunc=["sum", "size"], fill_value=0
    ).round(1)
)

## 5 · Stress (Table 5, Fig 5a, Fig 5b)

The same frozen book under conditions that should dent it, not reverse it:

* **pessimistic fills** — every buy at the fill bar's high, every sell at its low (`fill="pessimistic"`; the
  buy-and-hold hurdle is filled the same way). An intraday edge of a few bp that lives inside the open-fill
  assumption dies here;
* **2× costs** — commission, its minimum, spread and slippage all doubled (`COST_MULT`): 28 bp a round trip instead
  of 14. This one is a gate (§8);
* **no costs** — the gross book: how much of the result the costs are;
* **the health gate off / on** (`health_days` 0 vs the configured window) — does the regime brake help, or does it
  only trade less? A book that loses on its trades "improves" by trading less, so compare `bp_per_trade` (the mean
  net return of a trade, after costs), not only alpha. **Fig 5b** draws what the gate reads: the model's own rank IC
  over the previous `health_days` sessions (completed outcomes only), and the line above which a session is traded;
* **long only / short only** — is one side carrying the book?

Every row is a full `bt.evaluate` on TEST, scored from its first bar, plus the mean net return of its trades. The
forecasts are the same in every row (cached), so only the trading changes.

In [ ]:
# stress(label, params, ...): one full bt.evaluate of the candidate on TEST under a changed
# condition, plus its mean net bp per trade (research.stress, bound to this candidate)
stress = functools.partial(
    rs.stress, base=PRM, panel=panel, start=TEST[0], strategy=strat, sett=settings
)
GATE_DAYS = PRM["health_days"] or 5
configured = "as configured" + (
    f" (health gate on, {GATE_DAYS} days)" if PRM["health_days"] else ""
)
cases = [
    stress(configured),
    stress("pessimistic fills", strategy=dataclasses.replace(strat, fill="pessimistic")),
    stress(f"{COST_MULT:g}x costs", sett=rs.with_costs(settings, COST_MULT)),
    stress("no costs (gross)", sett=rs.with_costs(settings, 0.0)),
    stress("health gate off", {"health_days": 0})
    if PRM["health_days"]
    else stress(f"health gate on ({GATE_DAYS} days)", {"health_days": GATE_DAYS}),
    stress("long only", {"sides": "long"}),
    stress("short only", {"sides": "short"}),
]
table5 = pd.DataFrame(cases).set_index("case")
display(table5.round(4))

a = table5["alpha_cagr"].iloc[::-1]
fig = go.Figure(
    go.Bar(
        x=a,
        y=a.index,
        orientation="h",
        marker_color=[plots.SERIES[0] if v >= 0 else plots._LOSS for v in a],
        text=[f"{v:+.1%}" for v in a],
        textposition="outside",
        showlegend=False,
    )
)
fig.add_vline(x=0, line={"color": plots._INK, "width": 1})
plots._base(fig, title="Fig 5a · alpha per year on TEST under each stress", height=380)
lo, hi = min(a.min(), 0.0), max(a.max(), 0.0)
pad = 0.2 * ((hi - lo) or 0.01)  # room for the labels outside the bars
fig.update_xaxes(tickformat="+.0%", range=[lo - pad, hi + pad])
fig.update_yaxes(automargin=True)
fig.show()

health_ic = pd.Series(SM["health"](fc, days=GATE_DAYS), index=pd.to_datetime(SESS))
gate_open = (health_ic > PRM["health_min"])[health_ic.index >= TEST[0]]
fig = go.Figure(
    plots._scatter(
        health_ic.dropna(),
        name=f"rank IC of the previous {GATE_DAYS} sessions",
        mode="lines+markers",
        line={"color": plots.SERIES[0], "width": 1.5},
        marker={"size": 4},
    )
)
fig.add_hline(
    y=PRM["health_min"],
    line={"color": plots._LOSS, "dash": "dash", "width": 1},
    annotation_text=f"health_min {PRM['health_min']:g}: traded above",
    annotation_position="bottom right",
)
fig.add_vrect(
    x0=TEST[0], x1=SESS[-1], fillcolor=COLOR["test"], opacity=0.08, line_width=0, layer="below"
)
plots._base(fig, title="Fig 5b · what the health gate reads (TEST shaded)", height=360)
fig.update_xaxes(type="date")
fig.update_yaxes(title_text=f"rank IC, previous {GATE_DAYS} sessions")
charts.below(fig, -0.15)
fig.show()
print(
    f"the gate is open on {gate_open.mean():.0%} of TEST sessions "
    f"({int(gate_open.sum())} of {len(gate_open)})"
    + ("" if PRM["health_days"] else " - it is OFF in this candidate")
)

## 6 · The coin-flip null on TEST (Table 6, Fig 6)

A day-clustered t tests against zero, but it assumes the machinery cannot invent information. The **coin-flip
market** (`rs.sign_surrogate`) checks that: every one-minute move of the real panel is mirrored up or down at random —
one coin per minute, shared by all five names so they still move together. Volatility, volume, the clock and the
option inputs stay as they were; direction is gone. `rs.null_study(..., window="test")` runs the whole walk-forward
on it, once per seed, and scores TEST exactly like §3: whatever it finds is what *no direction* looks like here.

The real TEST IC is judged against those runs with **`rs.null_z`**: (real IC − mean coin-flip IC) /
√(SE_real² + SE_null² / seeds), each SE the run's own day-clustered standard error (two seeds cannot give a spread of
their own), capped at the real IC's t against exactly 0 (a lucky low null mean must not lift it). §8 gates on z ≥
`MIN_NULL_Z`. The same trading rule also trades each coin-flip market (Table 6: alpha,
trades, net bp per trade): what the rule loses with nothing to find. Compare the books per trade, not by alpha: the
health gate opens on a different number of sessions in each market, and a book that trades less loses less.

**Cost.** One full walk-forward per seed on the TEST-end panel, about as long as §2's first run; afterwards the
forecasts are on disk (`cache/…-null<seed>-….pkl`, keyed by the coin-flip panel's content) and this section takes
seconds. `RUN_NULL = False` skips it, and the verdict is then INCOMPLETE. Two seeds is a small null: enough to show
that the machinery invents nothing and to anchor the z, not to place a 95th percentile.

In [ ]:
NULL, NULL_BOOK, Z_IC, Z_SIGN = None, None, np.nan, np.nan
if RUN_NULL:
    # one walk-forward per seed on the coin-flip copy of THIS panel: cache/...-null<seed>-...pkl
    NULL = rs.null_study(panel, PRM, seeds=NULL_SEEDS, window="test", cost_bp=RT_BP)
    Z_IC, Z_SIGN = rs.null_z(S_TEST, NULL, "ic"), rs.null_z(S_TEST, NULL, "sign_bp")
    books = {}
    for s in NULL_SEEDS:
        ps = rs.sign_surrogate(panel, s)
        rs.prime_cache(ps, PRM, extra=f"-null{s}")  # the coin-flip book trades Table 6's forecasts
        st = bt.evaluate(strat, ps, PRM, settings=settings, score_from=TEST[0])
        pf = bt.run_strategy(strat, ps, PRM, settings=settings, score_from=TEST[0])
        books[s] = {**st, "bp_per_trade": 1e4 * bt.trade_records(pf)["return"].mean()}
    NULL_BOOK = pd.DataFrame(books).T
    BOOK6 = ["alpha_cagr", "n_trades", "bp_per_trade"]
    KEYS = ["ic", "t", "hit", "auc", "mse_skill", "sign_bp", "sign_t"]
    table6 = pd.concat(
        [
            pd.DataFrame(
                {
                    **{k: S_TEST[k] for k in KEYS},
                    **{m: ST_BOOK[m] for m in ("alpha_cagr", "n_trades")},
                    "bp_per_trade": TRADES["bp"].mean(),
                },
                index=["real TEST"],
            ),
            NULL[KEYS].join(NULL_BOOK[BOOK6]).rename(index=lambda s: f"coin-flip seed {s}"),
        ]
    ).astype(float)
    display(table6.round(4))
    print(
        f"the real TEST against the coin-flip runs (rs.null_z): IC z {Z_IC:+.2f}, "
        f"sign_bp z {Z_SIGN:+.2f}"
    )

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("forecast rank IC, ±2 SE", "net bp per trade", "book alpha per year"),
    )
    colors = [COLOR["test"]] + [plots._MUTED] * len(NULL_SEEDS)
    se6 = (table6["ic"] / table6["t"]).abs()
    for j, col in enumerate(("ic", "bp_per_trade", "alpha_cagr"), start=1):
        err = {"type": "data", "array": 2 * se6, "color": plots._INK, "thickness": 1}
        fig.add_trace(
            go.Bar(
                x=table6.index,
                y=table6[col],
                marker_color=colors,
                error_y=err if col == "ic" else None,
                showlegend=False,
            ),
            row=1,
            col=j,
        )
        fig.add_hline(y=0, line={"color": plots._INK, "width": 1}, row=1, col=j)
    fig.update_yaxes(tickformat="+.0%", row=1, col=3)
    title = "Fig 6 · TEST, the real market against coin-flip markets (no direction)"
    plots._base(fig, title=title, height=400)
    fig.update_layout(margin={"t": 90})  # the subplot titles sit under the figure title
    fig.show()
else:
    print("coin-flip null NOT run (RUN_NULL = False)")

## 7 · The HOLDOUT (read once)

Off by default. Set `READ_HOLDOUT = True` only when TEST has given its verdict and nothing about the candidate will
change any more; run the notebook once and record the numbers in the README. The panel is then reloaded through the
last stored bar (`rs.load(..., end="all")`), the walk-forward continues through the HOLDOUT exactly as it did through
TEST (a model fit only on earlier sessions, refit on the configured cadence), and both the forecast and the book are
scored from `WINDOWS["holdout"][0]`. Re-reading it after a change turns the HOLDOUT into a second TEST window, and
there is no third one.

In [ ]:
S_HOLD, ST_HOLD, ST_HOLD_BASE = None, None, None
if READ_HOLDOUT:
    full = rs.load(settings, engine, end="all")  # every stored bar
    fc_hold = rs.prime_cache(full, PRM)  # the walk-forward through the last stored bar
    S_HOLD = rs.summary(rs.table(fc_hold, "holdout"), cost_bp=RT_BP)
    ST_HOLD = bt.evaluate(strat, full, PRM, settings=settings, score_from=HOLDOUT[0])
    ST_HOLD_BASE = bt.evaluate(baseline, full, {}, settings=settings, score_from=HOLDOUT[0])
    display(pd.DataFrame({"TEST": S_TEST, "HOLDOUT": S_HOLD}).astype(float).round(4))
    display(
        pd.DataFrame({"darts_forecast": ST_HOLD, "buy & hold (baseline)": ST_HOLD_BASE})
        .loc[COLS]
        .astype(float)
        .round(4)
    )
else:
    print(
        f"HOLDOUT NOT read (READ_HOLDOUT = False): nothing after {SESS[-1]} was loaded or scored."
    )

## 8 · Checks and the verdict

Three kinds of rows:

* **gate** — decides the verdict. All four are about TEST, and all must pass:
  1. the forecast carries information: rank IC > 0 with a day-clustered t ≥ `MIN_IC_T`;
  2. it beats the coin-flip null: `rs.null_z` of the IC ≥ `MIN_NULL_Z`;
  3. the book earns alpha after costs;
  4. it still does at 2× costs.
* **machinery** — must pass, or no number above can be trusted: no HOLDOUT bar was loaded, the forecast is this
  panel's with the candidate's knobs, TEST changes nothing in TUNE, every forecast comes from the latest fit made at
  or before its decision bar, every TRAIN window is the `train_days` sessions before its retrain point, and the
  coin-flip market shows no IC (|t| < 3: a leak shows as a t far beyond chance, while |t| < 2 would fail one seed in
  twenty by luck alone). A machinery failure makes the verdict **INVALID**, whatever the gates say.
* **warning** — reported, not decisive: pessimistic fills, the coin-flip books (per trade), the trade floor
  (`optuna.min_trades_per_year`), excess over buy-and-hold, the IC month by month, each side alone, and whether the
  health gate picks *better* trades (net bp per trade with it ≥ without it; a gate that only trades less is not a
  regime brake).

The verdict is **PASS** (every gate passes), **FAIL** (a gate fails), **INCOMPLETE** (a gate or machinery row was not
run) or **INVALID** (a machinery row fails). TEST is read here for the second time (see the top), so even a PASS is
weaker evidence than a first read would have been; the HOLDOUT is the one untouched window. `SCORECARD` is one
machine-readable line with the headline numbers, for the README and later comparisons.

In [ ]:
floor = strat.optuna.get("min_trades_per_year", 0)
pess, costly = table5.loc["pessimistic fills"], table5.loc[f"{COST_MULT:g}x costs"]
side = table5.loc[["long only", "short only"], "alpha_cagr"]
gate_row = [c for c in table5.index if c.startswith("health gate")][0]
gate_on, gate_off = (
    (table5.iloc[0], table5.loc[gate_row])
    if PRM["health_days"]
    else (table5.loc[gate_row], table5.iloc[0])
)
test_months = MONTHS[MONTHS["window"] == "TEST"]
# every forecast comes from the LATEST fit at or before its decision bar (never a later one)
rows_ = np.nonzero(np.isfinite(fc["edge"]))
fit_of = fc["fit"][rows_].astype(int)
at = np.searchsorted(LOG["retrain_bucket"].to_numpy(), rows_[0], side="right") - 1
CAUSAL_OK = bool((at >= 0).all() and (LOG["fit"].to_numpy()[np.maximum(at, 0)] == fit_of).all())
# every TRAIN window starts train_days sessions before its retrain session and ends just before
# the retrain point (before the VALIDATION slice for tide / tft)
where = {d: i for i, d in enumerate(SESS)}
n_train = LOG["session"].map(where) - LOG["train_from"].map(where)
gap = LOG["session"].map(where) - LOG["train_last"].map(where)
at_open = LOG["retrain_bucket"].isin(fc["inputs"]["g"]["sess_first"]).to_numpy()
want_gap = np.where(at_open | (VAL > 0), 1 + VAL, 0)  # an intraday retrain also fits its morning
WINDOW_OK = bool((n_train == PRM["train_days"]).all() and (gap == want_gap).all())
if NULL is not None:
    null_ok = bool(np.isfinite(Z_IC) and Z_IC >= MIN_NULL_Z)
    null_detail = (
        f"z {Z_IC:+.2f}: TEST IC {S_TEST['ic']:+.4f} (t {S_TEST['t']:+.2f}) vs coin-flip "
        + ", ".join(f"{v:+.4f}" for v in NULL["ic"])
    )
    book_null_ok = bool(TRADES["bp"].mean() > NULL_BOOK["bp_per_trade"].max())
    book_null_detail = (
        f"{TRADES['bp'].mean():+.1f} bp x {len(TRADES)} trades vs coin-flip "
        + ", ".join(
            f"{b:+.1f} bp x {int(n)}"
            for b, n in zip(NULL_BOOK["bp_per_trade"], NULL_BOOK["n_trades"], strict=True)
        )
        + f"; alpha {rs.pct(ST_BOOK['alpha_cagr'])} vs "
        + ", ".join(rs.pct(v) for v in NULL_BOOK["alpha_cagr"])
    )
    quiet_null = bool((NULL["t"].abs() < 3).all())  # a leak shows as a t far beyond chance
    quiet_detail = "coin-flip IC t: " + ", ".join(f"{v:+.2f}" for v in NULL["t"])
else:
    null_ok = book_null_ok = quiet_null = None
    null_detail = book_null_detail = quiet_detail = "not run (RUN_NULL = False)"
checks = [
    (
        "gate",
        f"TEST forecast: rank IC > 0 with day-clustered t >= {MIN_IC_T:g}",
        bool(S_TEST["ic"] > 0 and S_TEST["t"] >= MIN_IC_T),
        f"IC {S_TEST['ic']:+.4f}, t {S_TEST['t']:+.2f}, "
        f"{S_TEST['n']:,} forecasts on {S_TEST['days']} days",
    ),
    (
        "gate",
        f"TEST forecast beats the coin-flip null (rs.null_z of the IC >= {MIN_NULL_Z:g})",
        null_ok,
        null_detail,
    ),
    (
        "gate",
        "TEST book: alpha > 0 after costs",
        bool(ST_BOOK["alpha_cagr"] > 0),
        f"alpha {rs.pct(ST_BOOK['alpha_cagr'])}/yr, CAGR {rs.pct(ST_BOOK['cagr'])}, "
        f"beta {ST_BOOK['beta']:+.2f}, {ST_BOOK['n_trades']} trades",
    ),
    (
        "gate",
        f"TEST book survives {COST_MULT:g}x costs (alpha > 0)",
        bool(costly["alpha_cagr"] > 0),
        f"alpha {rs.pct(costly['alpha_cagr'])}/yr at {COST_MULT * RT_BP:.0f} bp a round trip",
    ),
    (
        "machinery",
        "the panel stops at TEST's last day (no HOLDOUT bar loaded)",
        bool(SESS[-1] <= TEST[1]),
        f"last session {SESS[-1]}; TEST ends {TEST[1]}",
    ),
    (
        "machinery",
        "the forecast is this panel's, made with the candidate's forecast knobs",
        FC_OK,
        FC_DETAIL,
    ),
    (
        "machinery",
        "TEST data changes nothing in TUNE (inputs, fits, forecasts)",
        TRUNC_OK,
        TRUNC_DETAIL,
    ),
    (
        "machinery",
        "every forecast comes from the latest fit made at or before its decision bar",
        CAUSAL_OK,
        f"{len(fit_of):,} forecasts from {len(LOG)} fits",
    ),
    (
        "machinery",
        "every TRAIN window is the train_days sessions just before its retrain point",
        WINDOW_OK,
        f"{n_train.min()}–{n_train.max()} sessions in {len(LOG)} windows; "
        f"train_days {PRM['train_days']}",
    ),
    (
        "machinery",
        "the coin-flip market shows no IC (|t| < 3 on every seed)",
        quiet_null,
        quiet_detail,
    ),
    (
        "warning",
        "survives pessimistic fills (alpha > 0)",
        bool(pess["alpha_cagr"] > 0),
        f"alpha {rs.pct(pess['alpha_cagr'])}/yr",
    ),
    (
        "warning",
        "TEST trades beat the coin-flip books' (net bp per trade above every seed)",
        book_null_ok,
        book_null_detail,
    ),
    (
        "warning",
        f"trades often enough (>= {floor}/yr, optuna.min_trades_per_year)",
        bool(ST_BOOK["trades_per_year"] >= floor),
        f"{ST_BOOK['n_trades']} trades = {ST_BOOK['trades_per_year']:,.0f}/yr",
    ),
    (
        "warning",
        "TEST book beats buy-and-hold (excess > 0)",
        bool(ST_BOOK["excess_cagr"] > 0),
        f"excess {rs.pct(ST_BOOK['excess_cagr'])}/yr; buy-and-hold CAGR {rs.pct(ST_BASE['cagr'])}",
    ),
    (
        "warning",
        "forecast IC > 0 in every TEST month",
        bool((test_months["ic"] > 0).all()),
        ", ".join(f"{m} {v:+.4f}" for m, v in test_months["ic"].items()),
    ),
    (
        "warning",
        "both sides earn alpha on their own",
        bool((side > 0).all()),
        f"long only {rs.pct(side['long only'])}, short only {rs.pct(side['short only'])}",
    ),
    (
        "warning",
        "the health gate picks better trades (net bp per trade with it >= without it)",
        bool(gate_on["bp_per_trade"] >= gate_off["bp_per_trade"]),
        f"with: {gate_on['bp_per_trade']:+.1f} bp x {int(gate_on['n_trades'])} trades, alpha "
        f"{rs.pct(gate_on['alpha_cagr'])}; without: {gate_off['bp_per_trade']:+.1f} bp x "
        f"{int(gate_off['n_trades'])} trades, alpha {rs.pct(gate_off['alpha_cagr'])}",
    ),
]
if READ_HOLDOUT:
    checks += [
        (
            "gate",
            "HOLDOUT forecast: rank IC > 0",
            bool(S_HOLD["ic"] > 0),
            f"IC {S_HOLD['ic']:+.4f}, t {S_HOLD['t']:+.2f}",
        ),
        (
            "gate",
            "HOLDOUT book: alpha > 0 after costs",
            bool(ST_HOLD["alpha_cagr"] > 0),
            f"alpha {rs.pct(ST_HOLD['alpha_cagr'])}/yr",
        ),
    ]

table8 = rs.checks_table([c[1:] for c in checks])
table8.insert(0, "kind", [c[0] for c in checks])
table8["result"] = table8["ok"].map(rs.state)
with pd.option_context("display.max_colwidth", None):
    display(table8[["kind", "check", "result", "detail"]])

kind, result = table8["kind"], table8["result"]
broken = table8[(kind == "machinery") & (result == "fail")]
failed = table8[(kind == "gate") & (result == "fail")]
missing = table8[kind.isin(["gate", "machinery"]) & (result == "not run")]
if len(broken):
    VERDICT = "INVALID"
elif len(failed):
    VERDICT = "FAIL"
elif len(missing):
    VERDICT = "INCOMPLETE"
else:
    VERDICT = "PASS"
print(
    f"VERDICT on TEST (re-read after the core fixes): {VERDICT}"
    + ("" if READ_HOLDOUT else " · HOLDOUT not read")
)
for _, r in pd.concat([broken, failed]).iterrows():
    print(f"  failed  [{r['kind']}] {r['check']}: {r['detail']}")
for _, r in missing.iterrows():
    print(f"  not run [{r['kind']}] {r['check']}")
warned = table8[(kind == "warning") & (result == "fail")]
print(f"warnings: {len(warned)} of {int((kind == 'warning').sum())} flagged")

FC_KEYS = ("ic", "t", "hit", "auc", "mse_skill", "sign_bp", "sign_t", "n")
BOOK_KEYS = (
    "alpha_cagr",
    "cagr",
    "beta",
    "excess_cagr",
    "volatility",
    "max_drawdown",
    "n_trades",
    "trades_per_year",
)
rs.scorecard(
    {
        "notebook": "darts_forecast_diagnostics",
        "candidate": SOURCE,
        "overrides": OVERRIDES,
        "forecast": {
            k: PRM[k]
            for k in ("model", "bar_minutes", "horizon", "retrain_minutes", "train_days", "options")
        },
        "rule": {k: PRM[k] for k in ("edge_bp", "hold_bars", "stop_vol", "health_min")},
        "test": [TEST[0], SESS[-1]],
        "test_read": "second: re-read after the core fixes (first read 2026-10-02, old core)",
        "round_trip_bp": RT_BP,
        "forecast_tune": {k: S_TUNE[k] for k in FC_KEYS},
        "forecast_test": {k: S_TEST[k] for k in FC_KEYS},
        "null_ic": None if NULL is None else NULL["ic"].to_dict(),
        "null_z": {"ic": Z_IC, "sign_bp": Z_SIGN},
        "book_test": {k: ST_BOOK[k] for k in BOOK_KEYS},
        "buy_hold_test_cagr": ST_BASE["cagr"],
        "stress_alpha": table5["alpha_cagr"].to_dict(),
        "stress_bp_per_trade": table5["bp_per_trade"].to_dict(),
        "null_alpha": None if NULL_BOOK is None else NULL_BOOK["alpha_cagr"].to_dict(),
        "null_bp_per_trade": None if NULL_BOOK is None else NULL_BOOK["bp_per_trade"].to_dict(),
        "truncation_identical": TRUNC_OK,
        "holdout_read": READ_HOLDOUT,
        "forecast_holdout": None if S_HOLD is None else {k: S_HOLD[k] for k in FC_KEYS},
        "book_holdout": None if ST_HOLD is None else {k: ST_HOLD[k] for k in BOOK_KEYS},
        "gates_passed": f"{int(((kind == 'gate') & (result == 'pass')).sum())}"
        f"/{int((kind == 'gate').sum())}",
        "checks_passed": f"{int((result == 'pass').sum())}/{len(table8)}",
        "verdict": VERDICT,
    }
)